In [0]:
%run ../../common/_utils_config

In [0]:
%run ../../common/_utils_reference_data

In [0]:
%run ../utils/_utils_silver_contracts

In [0]:
import copy

failures = []


def check(description, condition):
    print(("✅ " if condition else "❌ ") + description)
    if not condition:
        failures.append(description)


def has_error(errors, text):
    return any(text in e for e in errors)


contracts = load_all_entity_contracts()
print("Contracts found:", list(contracts))

contract_errors = validate_all_entity_contracts(contracts)
for name, errors in contract_errors.items():
    print(f"   {name}: {errors}")

check("expected contracts exist",
      {"data_source", "location", "electricity_price_hourly", "weather_observation_hourly"} <= set(contracts))
check("all contracts in domain_model/entities are valid", not contract_errors)

if contract_errors:
    raise AssertionError("Fix the contracts above before running the next cells")

In [0]:
def broken(entity_name, change):
    """Validates a modified copy of a contract."""
    contract = copy.deepcopy(contracts[entity_name])
    change(contract)
    return validate_entity_contract(contract, contracts)


def set_column(contract, column_name, **changes):
    for column in contract["columns"]:
        if column["name"] == column_name:
            column.update(changes)


weather, location, data_source = "weather_observation_hourly", "location", "data_source"

cases = [
    ("unknown business key", weather,
     lambda c: c.update(business_keys=["province_code", "missing_column"]), "is not a column"),
    ("nullable business key", weather,
     lambda c: set_column(c, "province_code", nullable=True), "cannot be nullable"),
    ("double as business key", weather,
     lambda c: c.update(business_keys=["province_code", "datetime_utc", "temperature_c"]), "cannot be of type"),
    ("FK to an entity without contract", weather,
     lambda c: c.update(references=[{"entity": "unknown_entity", "columns": ["province_code"]}]), "has no contract"),
    ("FK to an entity that is not a reference", weather,
     lambda c: c.update(references=[{"entity": "electricity_price_hourly", "columns": ["province_code"]}]),
     "can only point to reference entities"),
    ("FK to data_source declared by hand", weather,
     lambda c: c.update(references=[{"entity": "data_source", "columns": ["province_code"]}]),
     "is added automatically"),
    ("FK with wrong number of columns", weather,
     lambda c: c.update(references=[{"entity": "location", "columns": ["province_code", "datetime_utc"]}]),
     "must match the business keys"),
    ("FK with a different type", weather,
     lambda c: set_column(c, "province_code", type="int"), "does not match the type"),
    ("invalid column type", weather,
     lambda c: set_column(c, "temperature_c", type="float"), "type must be one of"),
    ("accepted_values on a numeric column", weather,
     lambda c: set_column(c, "temperature_c", accepted_values=["1"]), "accepted_values only applies to string"),
    ("range on a string column", weather,
     lambda c: set_column(c, "province_code", range={"min": 0}), "range only applies to numeric"),
    ("framework column defined in an entity", weather,
     lambda c: c["columns"].append({"name": "source_name", "type": "string", "description": "x"}),
     "is added by the framework"),
    ("completeness grouped by an unknown column", weather,
     lambda c: c["completeness"].update(group_by=["missing_column"]), "completeness.group_by"),
    ("source_priority (left for MDM)", weather,
     lambda c: c.update(source_priority=["open_meteo_archive"]), "unknown keys"),
    ("entity name with the reserved golden suffix", weather,
     lambda c: c.update(entity=f"weather{golden_suffix}"), "reserved for MDM golden record tables"),
    ("reference with two origins", location,
     lambda c: c.update(reference_source="source_specs"), "exactly one of reference_file"),
    ("data_source keyed by another column", data_source,
     lambda c: c.update(business_keys=["provider"]), "business_keys must be [source_name]"),
    ("data_source with a field the framework does not provide", data_source,
     lambda c: c["columns"].append({"name": "owner", "type": "string", "description": "x"}),
     "are not data_source fields"),
]

for description, entity_name, change, expected in cases:
    errors = broken(entity_name, change)
    check(f"rejects {description}", has_error(errors, expected))
    if not has_error(errors, expected):
        print(f"   got: {errors}")

In [0]:
for name in ("weather_observation_hourly", "location", "data_source"):
    print(describe_silver_tables(contracts[name]))
    print()

weather_tables = build_silver_tables(contracts["weather_observation_hourly"])
weather_table = weather_tables[0]
weather_columns = [c for c, _ in weather_table["columns"]]
weather_fks = {fk["name"]: fk for fk in weather_table["foreign_keys"]}
location_table = build_silver_tables(contracts["location"])[0]
location_columns = [c for c, _ in location_table["columns"]]
data_source_table = build_silver_tables(contracts["data_source"])[0]

check("weather contract produces a single table", [t["name"] for t in weather_tables] == ["weather_observation_hourly"])
check("PK is the first column", weather_columns[0] == "PKWeatherObservationHourly")
check("entity PK = business keys + source_name",
      weather_table["primary_key"]["columns"] == ["province_code", "datetime_utc", "source_name"])
check("FKDataSource and FKLocation come right after the PK", weather_columns[1:3] == ["FKDataSource", "FKLocation"])
check("entity FKDataSource = md5(source_name) -> data_source.PKDataSource",
      weather_fks["FKDataSource"]["columns"] == ["source_name"]
      and weather_fks["FKDataSource"]["references_column"] == "PKDataSource")
check("FKLocation hashes the same columns as PKLocation",
      weather_fks["FKLocation"]["columns"] == location_table["primary_key"]["columns"])
check("location points to its data source 'locations_es'",
      location_table["foreign_keys"][0]["name"] == "FKDataSource"
      and location_table["foreign_keys"][0]["constant"] == "locations_es")
check("data_source points to its own row 'source_specs'",
      data_source_table["foreign_keys"][0]["constant"] == "source_specs"
      and data_source_table["foreign_keys"][0]["references_table"] == "data_source")
check("reference tables have no source columns", "source_endpoint" not in location_columns)
check("entity keeps the origin of each record",
      {"source_name", "source_endpoint", "source_record_id", "source_file"} <= set(weather_columns))
check("no table has duplicated column names",
      all(len({c for c, _ in t["columns"]}) == len(t["columns"])
          for contract in contracts.values() for t in build_silver_tables(contract)))

# data_source rows
rows = build_data_source_rows(contracts)
print("\ndata_source rows:")
for row in rows:
    print(f"  {row['source_name']:<22} {row['source_type']:<18} {row.get('provider') or '—':<40} {row.get('license') or '—'}")
rows_by_name = {r["source_name"]: r for r in rows}
bronze_names = {s["source_name"] for s in list_bronze_source_specs()}

check("data_source has every bronze source spec", bronze_names <= set(rows_by_name))
check("data_source has the reference file locations_es",
      rows_by_name.get("locations_es", {}).get("source_type") == "reference_file")
check("data_source has the platform row source_specs",
      rows_by_name.get("source_specs", {}).get("source_type") == "platform_metadata")
check("data source names are unique", len(rows_by_name) == len(rows))

missing_metadata = [r["source_name"] for r in rows if not r.get("provider") or not r.get("license")]
if missing_metadata:
    print(f"⚠️ provider or license missing for: {missing_metadata} (add them to the source specs; not an error)")

In [0]:
erd = build_erd_mermaid(contracts, focus=["weather_observation_hourly"], highlight=["weather_observation_hourly"])
print(erd)

check("ERD shows weather -> location", 'location ||--o{ weather_observation_hourly : "FKLocation"' in erd)
check("ERD shows weather -> data_source", 'data_source ||--o{ weather_observation_hourly : "FKDataSource"' in erd)
check("ERD shows location -> data_source", 'data_source ||--o{ location : "FKDataSource"' in erd)
check("ERD does not draw the data_source self-reference", 'data_source ||--o{ data_source' not in erd)
check("ERD marks weather as new", "class weather_observation_hourly entity_new" in erd)
check("ERD focus hides unrelated entities", "electricity_price_hourly" not in erd)
check("ERD hides lineage columns", "bronze_run_id" not in erd)

display_erd(erd)

In [0]:
display_erd(build_erd_mermaid(contracts))

doc_path = write_data_model_doc(contracts)
with open(doc_path, encoding="utf-8") as f:
    doc_text = f.read()
check("data model doc written with the diagram", "```mermaid" in doc_text and "PKLocation" in doc_text)
print(f"\n📄 {doc_path}")

if failures:
    raise AssertionError(f"{len(failures)} check(s) failed: {failures}")
print("\n✅ All silver contract checks passed")